# LVIS Lidar Mission Planning

This notebook demonstrates how to plan missions for the **LVIS (Land, Vegetation,
and Ice Sensor)** full-waveform scanning lidar using HyPlan.

We cover:

1. LVIS sensor fundamentals (scanner geometry, laser footprint)
2. Lens options and their effect on coverage
3. Contiguous vs. non-contiguous coverage
4. Effective swath width as a function of altitude and speed
5. Swath polygon generation for flight planning
6. Mission planning with aircraft integration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from hyplan.lvis import (
    LVIS, LVISLens,
    LVIS_LENS_NARROW, LVIS_LENS_MEDIUM, LVIS_LENS_WIDE, LVIS_LENSES,
)
from hyplan.aircraft import NASA_P3, NASA_B777
from hyplan.flight_line import FlightLine
from hyplan.swath import generate_swath_polygon, calculate_swath_widths
from hyplan.units import ureg

## 1. LVIS Sensor Overview

LVIS is a full-waveform scanning lidar operated by NASA Goddard Space Flight Center.
The scanner has a fixed maximum swath of 0.2 x altitude (half-scan angle ~ 5.7 deg).
Whether that swath is filled contiguously depends on:

- **Laser repetition rate** (default 4000 Hz)
- **Lens divergence** (determines footprint diameter)
- **Aircraft ground speed**

In [ ]:
lvis = LVIS()  # Default: 4000 Hz rep rate, wide lens

print(f"LVIS Default Configuration:")
print(f"  Rep rate:       {lvis.rep_rate}")
print(f"  Lens:           {lvis.lens.name} ({lvis.lens.divergence_mrad:.3f} mrad)")
print(f"  Half-scan angle: {lvis.half_angle:.2f} deg")

## 2. Lens Comparison

LVIS has three standard lens options. A wider divergence lens produces a
larger footprint, which helps fill the swath at higher speeds or altitudes.

In [ ]:
rows = []
for name, lens in LVIS_LENSES.items():
    rows.append({
        "Lens": name,
        "Divergence (mrad)": lens.divergence_mrad,
    })

lens_df = pd.DataFrame(rows)
lens_df

In [ ]:
# Footprint diameter vs altitude for each lens
altitudes_m = np.arange(2000, 12001, 500)

fig, ax = plt.subplots(figsize=(9, 5))

for name, lens in LVIS_LENSES.items():
    footprints = [
        lens.footprint_diameter(ureg.Quantity(a, "meter")).magnitude
        for a in altitudes_m
    ]
    ax.plot(altitudes_m / 1000, footprints, "o-", markersize=3, label=f"{name} ({lens.divergence_mrad} mrad)")

ax.set_xlabel("Altitude AGL (km)")
ax.set_ylabel("Footprint Diameter (m)")
ax.set_title("LVIS Laser Footprint vs. Altitude")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Maximum vs. Effective Swath Width

The **maximum swath** is set by scanner geometry (0.2 x altitude). The
**effective swath** may be narrower if the laser footprint is too small
to tile the full swath at a given speed.

In [ ]:
altitude = ureg.Quantity(8000, "meter")
speeds_kt = np.arange(100, 301, 10)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, lens in LVIS_LENSES.items():
    lvis_cfg = LVIS(lens=name)
    max_swaths = []
    eff_swaths = []
    contiguous_flags = []

    for spd in speeds_kt:
        speed = ureg.Quantity(spd, "knot")
        ms = lvis_cfg.swath_width(altitude).to(ureg.meter).magnitude
        es = lvis_cfg.effective_swath_width(altitude, speed).to(ureg.meter).magnitude
        max_swaths.append(ms)
        eff_swaths.append(es)
        contiguous_flags.append(lvis_cfg.is_contiguous(altitude, speed))

    ax = axes[list(LVIS_LENSES.keys()).index(name)]
    ax.plot(speeds_kt, np.array(max_swaths) / 1000, "k--", alpha=0.5, label="Max swath")
    ax.plot(speeds_kt, np.array(eff_swaths) / 1000, "b-", linewidth=2, label="Effective swath")

    # Shade contiguous region
    cont_speeds = [s for s, c in zip(speeds_kt, contiguous_flags) if c]
    if cont_speeds:
        ax.axvspan(speeds_kt[0], max(cont_speeds), alpha=0.1, color="green", label="Contiguous")

    ax.set_xlabel("Aircraft Speed (knots)")
    ax.set_ylabel("Swath Width (km)")
    ax.set_title(f"{name} lens")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Effective Swath Width at {altitude.to(ureg.km):.0f} AGL", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Coverage Summary

The `summary()` and `print_summary()` methods provide a complete
set of coverage parameters for a given flight configuration.

In [ ]:
altitude = ureg.Quantity(8000, "meter")
speed = ureg.Quantity(200, "knot")

lvis_wide = LVIS(lens="wide")
lvis_wide.print_summary(altitude, speed)

In [ ]:
# Compare all lenses at once
lvis_wide.compare_lenses(altitude, speed)

## 5. Effective FOV

When coverage is not contiguous, the effective FOV narrows.
Let's see how effective FOV varies with speed for different lenses.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for name in LVIS_LENSES:
    lvis_cfg = LVIS(lens=name)
    fovs = [
        lvis_cfg.effective_fov(altitude, ureg.Quantity(s, "knot"))
        for s in speeds_kt
    ]
    ax.plot(speeds_kt, fovs, "o-", markersize=3, label=f"{name} lens")

ax.axhline(y=2 * lvis.half_angle, color="red", linestyle="--", alpha=0.5, label="Max FOV")
ax.set_xlabel("Aircraft Speed (knots)")
ax.set_ylabel("Effective FOV (deg)")
ax.set_title(f"Effective FOV at {altitude.to(ureg.km):.0f} AGL")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Swath Polygon Generation

LVIS implements the standard `Sensor` interface (`swath_width`, `half_angle`),
so it works directly with `generate_swath_polygon` and flight box generation.

In [ ]:
fl = FlightLine.center_length_azimuth(
    lat=18.3, lon=-65.8,
    length=ureg.Quantity(40, "km"),
    az=90.0,
    altitude_msl=ureg.Quantity(28000, "feet"),
    site_name="LVIS Caribbean",
)

lvis_sensor = LVIS(lens="wide")
swath = generate_swath_polygon(fl, lvis_sensor)
widths = calculate_swath_widths(swath)

print(f"LVIS swath for {fl.site_name}:")
print(f"  Min width:  {widths['min_width']:.0f} m")
print(f"  Mean width: {widths['mean_width']:.0f} m")
print(f"  Max width:  {widths['max_width']:.0f} m")

# Theoretical max swath
theoretical = lvis_sensor.swath_width(fl.altitude_msl).to(ureg.meter).magnitude
print(f"  Theoretical max: {theoretical:.0f} m")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
gpd.GeoSeries([swath]).plot(ax=ax, alpha=0.3, color="forestgreen", edgecolor="darkgreen")
x, y = fl.geometry.xy
ax.plot(x, y, "r-", linewidth=2, label="Flight track")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"LVIS Swath — {fl.site_name}")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 7. Altitude and Speed Trade-offs

LVIS mission planning involves balancing:
- **Higher altitude** = wider swath (fewer lines) but larger footprint needed for contiguous coverage
- **Slower speed** = easier to achieve contiguous coverage but less area covered per hour

In [ ]:
alts = np.arange(3000, 11001, 1000)
speeds = [150, 200, 250]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lvis_w = LVIS(lens="wide")

for spd in speeds:
    speed = ureg.Quantity(spd, "knot")
    eff = [lvis_w.effective_swath_width(ureg.Quantity(a, "meter"), speed).magnitude / 1000 for a in alts]
    mx = [lvis_w.swath_width(ureg.Quantity(a, "meter")).magnitude / 1000 for a in alts]
    cov = [lvis_w.coverage_rate(ureg.Quantity(a, "meter"), speed).magnitude / 1e6 for a in alts]

    axes[0].plot(alts / 1000, eff, "o-", markersize=4, label=f"{spd} kt")
    axes[1].plot(alts / 1000, cov, "s-", markersize=4, label=f"{spd} kt")

# Add max swath line
axes[0].plot(alts / 1000, mx, "k--", alpha=0.4, label="Max swath")

axes[0].set_xlabel("Altitude AGL (km)")
axes[0].set_ylabel("Effective Swath (km)")
axes[0].set_title("Effective Swath Width (wide lens)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Altitude AGL (km)")
axes[1].set_ylabel("Coverage Rate (km²/s)")
axes[1].set_title("Area Coverage Rate")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Custom Repetition Rate

LVIS can operate at different repetition rates. Higher rep rates
improve contiguous coverage at higher speeds.

In [ ]:
altitude = ureg.Quantity(8000, "meter")
speed = ureg.Quantity(250, "knot")

rep_rates = [2000, 4000, 8000, 10000]

rows = []
for rr in rep_rates:
    lvis_rr = LVIS(rep_rate=rr * ureg.Hz, lens="wide")
    s = lvis_rr.summary(altitude, speed)
    rows.append({
        "Rep Rate (Hz)": rr,
        "Footprint (m)": f"{s['footprint_diameter'].magnitude:.1f}",
        "Max Swath (m)": f"{s['max_swath'].magnitude:.0f}",
        "Eff Swath (m)": f"{s['effective_swath_width'].magnitude:.0f}",
        "Contiguous": s['contiguous'],
    })

rr_df = pd.DataFrame(rows)
print(f"LVIS coverage at {altitude.to(ureg.km):.0f} AGL, {speed} (wide lens):\n")
rr_df

## Summary

| Feature | Class/Function | Purpose |
|---------|---------------|----------|
| LVIS sensor | `LVIS(rep_rate, lens)` | Configure laser scanner |
| Lens options | `LVIS_LENS_NARROW/MEDIUM/WIDE` | Control footprint size |
| Max swath | `swath_width(altitude)` | Geometric scanner limit |
| Effective swath | `effective_swath_width(altitude, speed)` | Contiguous-coverage limit |
| Coverage check | `is_contiguous(altitude, speed)` | Verify full swath is filled |
| FOV | `effective_fov(altitude, speed)` | Actual FOV accounting for coverage |
| Coverage rate | `coverage_rate(altitude, speed)` | Area per unit time |
| Summary | `print_summary()`, `compare_lenses()` | Quick diagnostics |